# Building a GPT from Scratch: Self-Attention and Transformers

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/05-Attention-and-GPT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Recommended environment: Google Colab with a T4 GPU (`Runtime → Change runtime type → T4 GPU`).

In [ ]:
# Download the reference solutions for this notebook's exercises (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_attention.py', 'q_topk.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)

## Data: Tiny Shakespeare

We will train a **character-level** language model: given the preceding characters, predict the next one.
This is the basic principle behind large language models (LLMs) such as ChatGPT, just at a much smaller scale.

In [ ]:
from urllib.request import urlopen
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = urlopen(url).read().decode("utf8")
print(len(text))
print(text[:300])

### Tokenizer

The simplest tokenizer: one ID per character. (Real LLMs use methods such as BPE, which we will see in the next unit.)

In [ ]:
chars = sorted(set(text))
vocab_size = len(chars)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for i, c in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join(itos[i] for i in l)
print(vocab_size, repr("".join(chars)))
print(encode("Hello"), decode(encode("Hello")))

In [ ]:
data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, batch_size, block_size):
    d = train_data if split == "train" else val_data
    ix = torch.randint(len(d) - block_size - 1, (batch_size,))
    x = torch.stack([d[i:i + block_size] for i in ix])
    y = torch.stack([d[i + 1:i + block_size + 1] for i in ix])   # the target is shifted by one position
    return x.to(device), y.to(device)

x, y = get_batch("train", 2, 16)
for t in range(5):
    print(repr(decode(x[0, :t + 1].tolist())), "->", repr(itos[y[0, t].item()]))

## Baseline: Bigram model

Predict the next character from the previous one only. The whole model is a `vocab_size x vocab_size` table (`nn.Embedding`).

In [ ]:
class Bigram(nn.Module):
    def __init__(self):
        super().__init__()
        self.table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None):
        logits = self.table(idx)                     # (B, T, vocab)
        loss = None if targets is None else F.cross_entropy(logits.view(-1, vocab_size), targets.view(-1))
        return logits, loss

@torch.no_grad()
def generate(model, idx, max_new_tokens, temperature=1.0, block_size=None):
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx if block_size is None else idx[:, -block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :].float() / temperature   # only the prediction at the last position
        probs = F.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)  # sample according to the probabilities
        idx = torch.cat([idx, next_id], dim=1)
    return idx

bigram = Bigram().to(device)
opt = torch.optim.AdamW(bigram.parameters(), lr=1e-2)
for step in range(3000):
    xb, yb = get_batch("train", 64, 32)
    _, loss = bigram(xb, yb)
    opt.zero_grad(); loss.backward(); opt.step()
print("loss", loss.item())
start = torch.tensor([encode("\n")], device=device)
print(decode(generate(bigram, start, 300)[0].tolist()))

## Self-Attention

The bigram model only looks at the previous character. We want each position to be able to "consult" information from **all** previous positions.

The simplest idea: average over all previous positions. With matrix multiplication plus a lower-triangular mask (causal mask) we can compute it all at once:

In [ ]:
T = 5
mask = torch.tril(torch.ones(T, T))
w = torch.zeros(T, T).masked_fill(mask == 0, float("-inf"))
w = F.softmax(w, dim=-1)
w

Averaging treats every position as equally important. **Attention** lets the data decide the weights instead:

* Each position produces three vectors: a query $q$, a key $k$, and a value $v$
* Weights = $\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}}\right)$ (plus a causal mask, so it can't peek at the future)
* Output = weights $\times V$

$$\mathrm{Attention}(Q,K,V) = \mathrm{softmax}\left(\frac{QK^T}{\sqrt{d}} + M\right)V$$

## Q
Implement causal attention with basic matrix operations and compare the result with PyTorch's built-in
`F.scaled_dot_product_attention(q, k, v, is_causal=True)`.

* `q, k, v` all have shape `(B, n_head, T, head_dim)`
* The built-in SDPA automatically picks the fastest implementation (FlashAttention, memory-efficient attention, etc.), so use it whenever you can

In [ ]:
# %load q_attention.py
%run -i q_attention.py

## Transformer Block

A block in a modern LLM (Llama, Qwen, ...) looks roughly like this:

```
x = x + Attention(RMSNorm(x))
x = x + MLP(RMSNorm(x))
```

* **Multi-head**: split the vector into several parts and run attention on each separately
* **Residual connection** + **Pre-normalization**: make deep networks easy to train
* Real LLMs add improvements such as RoPE positional encoding, SwiGLU, and GQA; here we start with the simplest version

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.n_head = n_head
        self.qkv = nn.Linear(n_embd, 3 * n_embd, bias=False)
        self.proj = nn.Linear(n_embd, n_embd, bias=False)
        self.dropout = dropout
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        # (B, T, C) -> (B, n_head, T, head_dim)
        q, k, v = [t.view(B, T, self.n_head, C // self.n_head).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.dropout if self.training else 0.0)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.proj(y)

class MLP(nn.Module):
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd, bias=False), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd, bias=False), nn.Dropout(dropout))
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.norm1 = nn.RMSNorm(n_embd)
        self.attn = CausalSelfAttention(n_embd, n_head, dropout)
        self.norm2 = nn.RMSNorm(n_embd)
        self.mlp = MLP(n_embd, dropout)
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.1):
        super().__init__()
        self.block_size = block_size
        self.tok_emb = nn.Embedding(vocab_size, n_embd)
        self.pos_emb = nn.Embedding(block_size, n_embd)   # learnable positional encoding
        self.blocks = nn.Sequential(*[Block(n_embd, n_head, dropout) for _ in range(n_layer)])
        self.norm = nn.RMSNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init_weights)
    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        x = self.blocks(x)
        logits = self.lm_head(self.norm(x))
        loss = None if targets is None else F.cross_entropy(logits.view(-1, logits.size(-1)).float(), targets.view(-1))
        return logits, loss

In [ ]:
block_size = 256
model = GPT(vocab_size, block_size, n_layer=6, n_head=6, n_embd=384, dropout=0.2).to(device)
print(f"parameters: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## Training

We use fp16 mixed precision. It takes about 3 minutes on a T4.

Watch the `train` and `val` loss: if train keeps dropping while val starts rising, that is **overfitting** (the model starts memorizing the training data). Try increasing `max_steps` to 3000, or setting `dropout` to 0.0. What happens?

In [ ]:
@torch.no_grad()
def estimate_loss(model, iters=20, batch_size=64):
    model.eval()
    out = {}
    for split in ["train", "val"]:
        losses = []
        for _ in range(iters):
            xb, yb = get_batch(split, batch_size, block_size)
            with torch.autocast(device, dtype=torch.float16):
                _, loss = model(xb, yb)
            losses.append(loss.item())
        out[split] = sum(losses) / len(losses)
    model.train()
    return out

max_steps, batch_size, lr = 1500, 64, 1e-3
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.1)
scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=max_steps, pct_start=0.05)
scaler = torch.amp.GradScaler(device)
t0 = time.perf_counter()
model.train()
for step in range(max_steps + 1):
    if step % 250 == 0:
        losses = estimate_loss(model)
        print(f"step {step:5d}  train {losses['train']:.3f}  val {losses['val']:.3f}  {time.perf_counter()-t0:.0f}s")
    if step == max_steps:
        break
    xb, yb = get_batch("train", batch_size, block_size)
    with torch.autocast(device, dtype=torch.float16):
        _, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # gradient clipping
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

In [ ]:
start = torch.tensor([encode("ROMEO:\n")], device=device)
with torch.autocast(device, dtype=torch.float16):
    print(decode(generate(model, start, 500, temperature=0.8, block_size=block_size)[0].tolist()))

## Temperature

The lower the `temperature`, the more conservative the model (repetitive, boring); the higher, the more creative (but also the more likely to produce gibberish).

In [ ]:
for t in [0.3, 1.0, 1.5]:
    print(f"===== temperature={t}")
    with torch.autocast(device, dtype=torch.float16):
        print(decode(generate(model, start, 200, temperature=t, block_size=block_size)[0].tolist()))

## Q: Top-k sampling
Modify `generate` to sample only from the `k` most likely characters (set the other logits to `-inf`).

Hint: `torch.topk(logits, k)`

In [ ]:
# %load q_topk.py
%run -i q_topk.py

## Food for thought

* Right now, generating each new character recomputes the whole sequence. Real LLMs use a **KV cache** to store the keys and values computed so far (🤗 transformers, in the next unit, handles this for us)
* Our model has ~10M parameters and is character-level; Qwen3-0.6B has 600 million parameters, a vocabulary of about 150K, and was trained on 36 trillion tokens
* Try training on other text (e.g. Chinese novels, or your own code)